# 01 · RZLT Data Sampling & Symbology Prep

This notebook loads the national Residential Zoned Land Tax (RZLT) dataset, maps the six `ZONE_SYMBOLOGY` codes to readable labels, and builds a binary target:

- **1 = Included / Liable** (codes 1, 2, 5, 6)
- **0 = Excluded** (codes 3, 4)

## 1. Load National RZLT & Map All 6 Symbology Codes

In [10]:
import os
import gc
import geopandas as gpd
import pandas as pd
import numpy as np

# Load full National RZLT dataset
rzlt_path = "../Datasets/RZLT-Dataset.geojson"
gdf_rzlt = gpd.read_file(rzlt_path)

if gdf_rzlt.crs != "EPSG:2157":
    gdf_rzlt = gdf_rzlt.to_crs(epsg=2157)

# Map all 6 ZONE_SYMBOLOGY codes
symbology_map = {
    1: "Retained_Residential",
    2: "Retained_MixedUse",
    3: "Exclusion_Residential",
    4: "Exclusion_MixedUse",
    5: "Inclusion_Residential",
    6: "Inclusion_MixedUse"
}

gdf_rzlt = gdf_rzlt[gdf_rzlt['ZONE_SYMBOLOGY'].isin([1, 2, 3, 4, 5, 6])].copy()
gdf_rzlt['SYMBOLOGY_LABEL'] = gdf_rzlt['ZONE_SYMBOLOGY'].map(symbology_map)

# Binary Target: 1 = All Liable/Included (1, 2, 5, 6) | 0 = Excluded (3, 4)
gdf_rzlt['target'] = gdf_rzlt['ZONE_SYMBOLOGY'].isin([1, 2, 5, 6]).astype(int)
gdf_rzlt['DECISION_STATUS'] = np.where(gdf_rzlt['target'] == 1, 'Included_Liable', 'Excluded')

print("=== Full National Breakdown ===")
print(gdf_rzlt['SYMBOLOGY_LABEL'].value_counts())

=== Full National Breakdown ===
SYMBOLOGY_LABEL
Retained_Residential     253487
Retained_MixedUse          3234
Inclusion_Residential      2907
Exclusion_Residential       312
Exclusion_MixedUse          276
Inclusion_MixedUse          247
Name: count, dtype: int64


## 2. Stratified Sampling Before Heavy Geometry Calculations

Excluded parcels are rare (588 of ~260k), so all of them are kept and the Included/Liable pool is sampled while preserving its sub-type mix:

- **1:1 balanced** (588 + 588 = 1,176 rows)
- **10:1 stratified** (588 + 5,880 = 6,468 rows), closer to the real class imbalance

Sampling is done *before* any geometry calculations to keep memory use and runtime low.

In [11]:
# Separate all Excluded parcels (588 rows)
df_excl = gdf_rzlt[gdf_rzlt['target'] == 0].copy()
n_excl = len(df_excl)

# Stratified sample from Included/Liable (1, 2, 5, 6) preserving sub-type proportions
df_incl_pool = gdf_rzlt[gdf_rzlt['target'] == 1]

# 1) 1:1 Balanced Sample (588 Excluded + 588 Included = 1,176 rows)
df_incl_1to1 = (
    df_incl_pool.groupby('ZONE_SYMBOLOGY', group_keys=False)
    .apply(lambda x: x.sample(n=max(1, int(round(n_excl * len(x) / len(df_incl_pool)))), random_state=42))
)
# Trim or pad by 1-2 rows if rounding shifted exact count
df_incl_1to1 = df_incl_1to1.sample(n=n_excl, random_state=42)

# 2) 10:1 Representative Sample (588 Excluded + 5,880 Included = 6,468 rows)
n_incl_10to1 = n_excl * 10
df_incl_10to1 = (
    df_incl_pool.groupby('ZONE_SYMBOLOGY', group_keys=False)
    .apply(lambda x: x.sample(n=max(1, int(round(n_incl_10to1 * len(x) / len(df_incl_pool)))), random_state=42))
)
df_incl_10to1 = df_incl_10to1.sample(n=n_incl_10to1, random_state=42)

gdf_sample_1to1 = pd.concat([df_excl, df_incl_1to1], ignore_index=True)
gdf_sample_10to1 = pd.concat([df_excl, df_incl_10to1], ignore_index=True)

print(f"1:1 Stratified Sample : {len(gdf_sample_1to1):,} rows")
print(f"10:1 Stratified Sample: {len(gdf_sample_10to1):,} rows")

1:1 Stratified Sample : 1,176 rows
10:1 Stratified Sample: 6,468 rows


C:\Users\musab\AppData\Local\Temp\ipykernel_19940\3755386951.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=max(1, int(round(n_excl * len(x) / len(df_incl_pool)))), random_state=42))
C:\Users\musab\AppData\Local\Temp\ipykernel_19940\3755386951.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=max(1, int(round(n_incl_10to1 * len(x) / len(df_incl_pool))))

## 3. Clean Features, Compute Centroids, Export & Free RAM

- Fills missing or zero `SITE_AREA` from the geometry (hectares, EPSG:2157)
- Fills missing `ZONE_GZT` and `LOCAL_AUTHORITY` with `Unknown`
- Adds parcel centroids (`centroid_x`, `centroid_y`) for spatial features
- Drops slivers under 0.005 ha, which is why the saved files have **1,111** and **6,303** rows instead of 1,176 and 6,468
- Deletes the full national GeoDataFrame afterwards to free RAM

Outputs are saved to `Datasets/RZLT_Samples/`.

In [12]:
def clean_and_prepare(gdf):
    gdf = gdf.copy()
    computed_area_ha = gdf.geometry.area / 10000.0
    gdf['SITE_AREA'] = gdf['SITE_AREA'].fillna(computed_area_ha)
    gdf.loc[gdf['SITE_AREA'] <= 0, 'SITE_AREA'] = computed_area_ha

    gdf['ZONE_GZT'] = gdf['ZONE_GZT'].fillna("Unknown").astype(str).str.strip()
    gdf['LOCAL_AUTHORITY'] = gdf['LOCAL_AUTHORITY'].fillna("Unknown").astype(str).str.strip()

    gdf['centroid_x'] = gdf.geometry.centroid.x
    gdf['centroid_y'] = gdf.geometry.centroid.y

    # Drop tiny slivers (< 0.005 ha)
    gdf = gdf[gdf['SITE_AREA'] >= 0.005].copy()

    keep_cols = [
        'PARCEL_ID', 'LOCAL_AUTHORITY', 'ZONE_GZT', 'ZONE_ORIG',
        'DATE_ADDED', 'SITE_AREA', 'DECISION_STATUS', 'target',
        'centroid_x', 'centroid_y', 'geometry'
    ]
    return gdf[[c for c in keep_cols if c in gdf.columns]].reset_index(drop=True)

gdf_clean_1to1 = clean_and_prepare(gdf_sample_1to1)
gdf_clean_10to1 = clean_and_prepare(gdf_sample_10to1)

output_dir = "../Datasets/RZLT_Samples"
os.makedirs(output_dir, exist_ok=True)

path_1to1 = os.path.join(output_dir, "RZLT_AllCodes_Balanced_1to1.geojson")
path_10to1 = os.path.join(output_dir, "RZLT_AllCodes_Stratified_10to1.geojson")

gdf_clean_1to1.to_file(path_1to1, driver="GeoJSON")
gdf_clean_10to1.to_file(path_10to1, driver="GeoJSON")

print(f"Saved -> {path_1to1} ({len(gdf_clean_1to1):,} rows)")
print(f"Saved -> {path_10to1} ({len(gdf_clean_10to1):,} rows)")

# Free the 260k-row GeoDataFrame from memory immediately
del gdf_rzlt, df_incl_pool, gdf_sample_1to1, gdf_sample_10to1
gc.collect()
print("Memory cleared!")

Saved -> ../Datasets/RZLT_Samples\RZLT_AllCodes_Balanced_1to1.geojson (1,111 rows)
Saved -> ../Datasets/RZLT_Samples\RZLT_AllCodes_Stratified_10to1.geojson (6,303 rows)
Memory cleared!
